In [20]:
import bs4
from langsmith import Client
hub = Client()
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from chromadb.utils import embedding_functions
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv


load_dotenv()

#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Embed
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=embedding_function)

retriever = vectorstore.as_retriever()

#### RETRIEVAL and GENERATION ####

# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise."),
    ("user", "Context: {context}\n\nQuestion: {question}")
])

# LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # or "mixtral-8x7b-32768", "gemma2-9b-it"
    temperature=0,
)
# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Question
rag_chain.invoke("What is Task Decomposition?")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2910.30it/s]


'Task decomposition refers to the process of breaking down a complex task into smaller, more manageable subgoals or steps. It can be achieved through various methods, including using large language models (LLMs) with simple prompting, task-specific instructions, or human inputs. Additionally, approaches like LLM+P utilize external classical planners and the Planning Domain Definition Language (PDDL) to facilitate long-horizon planning.'